In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import KFold, train_test_split
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

# ── Load raw full dataset ────────────────────────────────────
df_raw = pd.read_csv("train_raw.csv")

# Also load my top50 train to get the feature names
df_top50_train = pd.read_csv("top50_train.csv")
top50_features = [c for c in df_top50_train.columns if c != "critical_temp"]

# ── Split raw data  80/20, random_state=42
df_train_raw, _ = train_test_split(
    df_raw, test_size=0.2, random_state=42
)
print(f"Training samples for BOTH models: {len(df_train_raw)}")
# Should be 17,010 ✅

# ── Prepare features ────────────────────────────────────────
# All 81 features (for Shams)
X_81 = df_train_raw.drop("critical_temp", axis=1).values
y    = df_train_raw["critical_temp"].values

# Top 50 SHAP features only (for my model)
X_50 = df_train_raw[top50_features].values

print(f"my model input  : {X_50.shape}")
print(f"Shams model input : {X_81.shape}\n")

# ── Model definitions ────────────────────────────────────────
def build_my_model():
    base = [
        ('ridge', Ridge(alpha=1.0, solver='lsqr')),
        ('lasso', Lasso(alpha=0.000599, max_iter=1000)),
        ('knn',   KNeighborsRegressor(n_neighbors=5, weights='distance',
                                       algorithm='brute', metric='manhattan')),
        ('svr',   SVR(C=126.8554, kernel='rbf', gamma=0.194308,
                      epsilon=0.000139, tol=0.0001)),
        ('mlp',   MLPRegressor(hidden_layer_sizes=(200,100), activation='tanh',
                               learning_rate_init=0.005, solver='adam',
                               batch_size=256, alpha=0.0007,
                               max_iter=1000, random_state=42))
    ]
    meta = RandomForestRegressor(n_estimators=300,
                                 random_state=42, n_jobs=-1)
    return StackingRegressor(estimators=base, final_estimator=meta, cv=5)


# ── 5-fold CV on SAME 17,010 samples, SAME folds ────────────
kf = KFold(n_splits=10, shuffle=True, random_state=42)

my_rmse_folds  = []
shams_rmse_folds = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_50)):
    print(f"Fold {fold+1}/10 ──────────────────────────────")

    # my MODEL — 50 features, StandardScaler
    X_tr_50,  X_val_50  = X_50[tr_idx],  X_50[val_idx]
    y_tr, y_val          = y[tr_idx],     y[val_idx]

    sc_my = StandardScaler()
    X_tr_50_sc  = sc_my.fit_transform(X_tr_50)
    X_val_50_sc = sc_my.transform(X_val_50)

    my_model = build_my_model()
    my_model.fit(X_tr_50_sc, y_tr)
    my_rmse = np.sqrt(np.mean((y_val - my_model.predict(X_val_50_sc))**2))
    my_rmse_folds.append(my_rmse)
    print(f"  my  RMSE: {my_rmse:.4f} K")


In [2]:
import numpy as np
from scipy import stats

# ─────────────────────────────────────────────────────────────────────────────
# SHAP-Only Improvement: Statistical Significance Test
# Comparison: SHAP+RF (50 features, RF meta-learner) vs Shams et al. baseline
# (81 features, RF meta-learner) — identical 10-fold CV folds, random_state=42
# ─────────────────────────────────────────────────────────────────────────────

shap_rf_folds = np.array([
    9.9257, 10.4358, 9.9585,
    9.6872,  9.7078, 9.6126,
   10.5928,  9.2526, 10.6626, 10.0923
])  # 10-fold RMSE: SHAP-selected 50 features + RF meta-learner

shams_folds = np.array([
   10.35782162, 10.75903560, 10.72593060,
   10.51406822, 10.94141393,  9.88964194,
   11.44077419, 10.14587360, 11.02113116, 10.76626399
])  # 10-fold RMSE: Shams et al. replicated (81 features + RF meta-learner)

# ── Fold-wise differences ─────────────────────────────────────────────────────
diffs = shap_rf_folds - shams_folds

# ── Descriptive statistics ────────────────────────────────────────────────────
mean_reduction = -diffs.mean()
ci             = stats.t.interval(0.95, df=9,
                                  loc=diffs.mean(),
                                  scale=stats.sem(diffs))
cohens_dz      = abs(diffs.mean()) / diffs.std(ddof=1)
wins           = int(np.sum(shap_rf_folds < shams_folds))

# ── Normality check on differences (Shapiro-Wilk) ────────────────────────────
sw_stat, sw_p  = stats.shapiro(diffs)

# ── Parametric test: Paired t-test ───────────────────────────────────────────
t_stat, t_p    = stats.ttest_rel(shap_rf_folds, shams_folds)

# ── Non-parametric test: Wilcoxon signed-rank ────────────────────────────────
w_stat, w_p    = stats.wilcoxon(shap_rf_folds, shams_folds)

# ─────────────────────────────────────────────────────────────────────────────
# RESULTS
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print("   SHAP-ONLY IMPROVEMENT: STATISTICAL SIGNIFICANCE TEST")
print("   SHAP+RF (50 features) vs Shams et al. Baseline (81 features)")
print("=" * 65)

print(f"\n{'─'*65}")
print("  FOLD-WISE RMSE (K)")
print(f"{'─'*65}")
print(f"  {'Fold':<6} {'SHAP+RF':>10} {'Shams':>10} {'Reduction':>12}")
print(f"  {'────':<6} {'───────':>10} {'─────':>10} {'─────────':>12}")
for i in range(10):
    print(f"  {i+1:<6} {shap_rf_folds[i]:>10.4f} {shams_folds[i]:>10.4f} {-diffs[i]:>+12.4f}")

print(f"\n  {'Mean':<6} {shap_rf_folds.mean():>10.4f} {shams_folds.mean():>10.4f} {mean_reduction:>+12.4f}")
print(f"  {'Std':<6} {shap_rf_folds.std(ddof=1):>10.4f} {shams_folds.std(ddof=1):>10.4f}")
print(f"  Fold wins (SHAP+RF < Shams): {wins}/10")

print(f"\n{'─'*65}")
print("  NORMALITY CHECK (Shapiro-Wilk on paired differences)")
print(f"{'─'*65}")
print(f"  W = {sw_stat:.4f},  p = {sw_p:.4f}")
if sw_p >= 0.05:
    print("  → Normality NOT violated (p ≥ 0.05) ✅")
    print("  → Paired t-test is the primary test")
else:
    print("  → Normality violated (p < 0.05) ⚠️")
    print("  → Wilcoxon signed-rank is the primary test")

print(f"\n{'─'*65}")
print("  PARAMETRIC TEST: Paired t-test")
print(f"{'─'*65}")
print(f"  t-statistic : {t_stat:.4f}")
print(f"  p-value     : {t_p:.4f}")
print(f"  df          : 9")

print(f"\n{'─'*65}")
print("  NON-PARAMETRIC TEST: Wilcoxon Signed-Rank")
print(f"{'─'*65}")
print(f"  W-statistic : {w_stat:.4f}")
print(f"  p-value     : {w_p:.4f}")

print(f"\n{'─'*65}")
print("  EFFECT SIZE & CONFIDENCE INTERVAL")
print(f"{'─'*65}")
print(f"  Mean RMSE reduction : {mean_reduction:.4f} K")
print(f"  95% CI              : [{-ci[1]:.4f}, {-ci[0]:.4f}] K")
print(f"  Cohen's dz          : {cohens_dz:.4f}")

print(f"\n{'─'*65}")
print("  VERDICT")
print(f"{'─'*65}")
if t_p < 0.01 and w_p < 0.01:
    print("  HIGHLY SIGNIFICANT at 99% confidence level ✅")
    print("  Both parametric and non-parametric tests agree.")
elif t_p < 0.05 and w_p < 0.05:
    print("  SIGNIFICANT at 95% confidence level ✅")
else:
    print("  NOT SIGNIFICANT ❌")

print("=" * 65)